In [2]:
import pandas as pd
import numpy as np

### load the csv

In [6]:
df = pd.read_csv("../data/raw/ufo_sighting_data.csv",
low_memory= False)

In [8]:
df.head()
df.info()
df.describe()
df.describe(include='object')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80332 entries, 0 to 80331
Data columns (total 11 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Date_time                        80332 non-null  object 
 1   city                             80332 non-null  object 
 2   state/province                   74535 non-null  object 
 3   country                          70662 non-null  object 
 4   UFO_shape                        78400 non-null  object 
 5   length_of_encounter_seconds      80332 non-null  object 
 6   described_duration_of_encounter  80332 non-null  object 
 7   description                      80317 non-null  object 
 8   date_documented                  80332 non-null  object 
 9   latitude                         80332 non-null  object 
 10  longitude                        80332 non-null  float64
dtypes: float64(1), object(10)
memory usage: 6.7+ MB


,Date_time,city,state/province,country,UFO_shape,length_of_encounter_seconds,described_duration_of_encounter,description,date_documented,latitude
count,80332,80332,74535,70662,78400,80332,80332,80317,80332,80332
unique,1390,19900,67,5,29,536,8090,79997,317,18421
top,10:00:00 PM,seattle,ca,us,light,300,5 minutes,Fireball,12/12/2009,47.6063889
freq,4617,525,9655,65114,16565,8635,4716,11,1510,581


In [9]:
df.isnull().sum()

Date_time                             0
city                                  0
state/province                     5797
country                            9670
UFO_shape                          1932
length_of_encounter_seconds           0
described_duration_of_encounter       0
description                          15
date_documented                       0
latitude                              0
longitude                             0
dtype: int64

# Initial Observations

- Dataset has 80332 rows.
- There are 11 columns.
- Latitude appears to be stored as text.
- Some columns contain missing values.
- Dates are stored as strings.

# Data Cleaning Strategy

Based on the initial observations, here's the approach to clean the UFO sighting dataset:

## Missing Values
- **state/province**: 5,797 missing values (~7%) - can be filled using geolocation or removed if context-dependent
- **country**: 9,670 missing values (~12%) - consider filling with 'us' if most sightings are US-based, or remove
- **UFO_shape**: 1,932 missing values (~2%) - can be categorized as 'unknown' or removed
- **description**: 15 missing values (<1%) - safe to remove

## Data Type Conversions
- **latitude**: Convert from object to float64 (currently stored as text)
- **Date_time** and **date_documented**: Convert to datetime format for time-series analysis
- **length_of_encounter_seconds**: Convert to numeric (likely contains mixed formats)

## Text Cleaning (UFO_shape)
- Standardize values (e.g., 'disc' vs 'disk', 'cigar' vs 'cylinder')
- Handle variations in capitalization and spacing
- Group rare categories into 'other' if needed

## Priority Order
1. Convert data types first
2. Handle missing values based on domain logic
3. Standardize categorical values (especially UFO_shape)
4. Validate latitude/longitude pairs for geographic accuracy

In [20]:
df.dtypes


Date_time                           object
city                                object
state/province                      object
country                             object
UFO_shape                           object
length_of_encounter_seconds         object
described_duration_of_encounter     object
description                         object
date_documented                     object
latitude                            object
longitude                          float64
dtype: object

In [17]:
df['latitude'].head(10)


0    29.8830556
1      29.38421
2          53.2
3    28.9783333
4    21.4180556
5        36.595
6     51.434722
7       41.1175
8    33.5861111
9    30.2947222
Name: latitude, dtype: object

In [18]:
df['length_of_encounter_seconds'].head(20)


0     2700
1     7200
2       20
3       20
4      900
5      300
6      180
7     1200
8      180
9      120
10     300
11     180
12    1800
13     180
14      30
15    1200
16     120
17    1800
18      20
19    2700
Name: length_of_encounter_seconds, dtype: object

In [19]:
df['length_of_encounter_seconds'].dtype

dtype('O')

In [22]:
df['country'].unique()

array(['us', nan, 'gb', 'ca', 'au', 'de'], dtype=object)

In [23]:
df['country'].nunique()

5

In [24]:
df['UFO_shape'].unique()

array(['cylinder', 'light', 'circle', 'sphere', 'disk', 'fireball',
       'unknown', 'oval', 'other', 'cigar', 'rectangle', 'chevron',
       'triangle', 'formation', nan, 'delta', 'changing', 'egg',
       'diamond', 'flash', 'teardrop', 'cone', 'cross', 'pyramid',
       'round', 'crescent', 'flare', 'hexagon', 'dome', 'changed'],
      dtype=object)

In [25]:
df['UFO_shape'].value_counts()

UFO_shape
light        16565
triangle      7865
circle        7608
fireball      6208
other         5649
unknown       5584
sphere        5387
disk          5213
oval          3733
formation     2457
cigar         2057
changing      1962
flash         1328
rectangle     1297
cylinder      1283
diamond       1178
chevron        952
egg            759
teardrop       750
cone           316
cross          233
delta            7
round            2
crescent         2
pyramid          1
flare            1
hexagon          1
dome             1
changed          1
Name: count, dtype: int64

In [26]:
df['country'].value_counts()

country
us    65114
ca     3000
gb     1905
au      538
de      105
Name: count, dtype: int64

In [27]:
df[df["country"].isnull()]

,Date_time,city,state/province,country,UFO_shape,length_of_encounter_seconds,described_duration_of_encounter,description,date_documented,latitude,longitude
1,9:00:00 PM,lackland afb,tx,NaN,light,7200,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.38421,-98.581082
18,11:00:00 PM,bermuda nas,NaN,NaN,light,20,20 sec.,saw fast moving blip on the radar scope thin w...,1/11/2002,32.364167,-64.678611
29,10:00:00 PM,saddle lake (canada),ab,NaN,triangle,270,4.5 or more min.,Lights far above&#44 that glance; then flee f...,1/19/2005,53.970571,-111.689885
35,7:00:00 AM,gisborne (new zealand),NaN,NaN,disk,120,2min,gisborne nz 1982 wainui beach to sponge bay,1/11/2002,-38.662334,178.017649
40,8:00:00 PM,holmes/pawling,ny,NaN,chevron,180,3 minutes,Football Field Sized Chevron with bright white...,10/8/2007,41.523427,-73.646795
...,...,...,...,...,...,...,...,...,...,...,...
80238,2:15:00 PM,broomfield?lafayette,co,NaN,rectangle,120,2 min,Large&#44 rectangular object seen flying in br...,12/12/2009,39.993596,-105.089706
80244,8:17:00 PM,lyman,me,NaN,light,600,10 mins,Two lights ran across the sky&#44 as bright as...,12/12/2009,43.505096,-70.637968
80319,8:15:00 PM,clifton,nj,NaN,other,3600,~1hr+,Luminous line seen in New Jersey sky.,9/30/2013,40.858433,-74.163755
80322,9:00:00 PM,aleksandrow (poland),NaN,NaN,light,15,15 seconds,Two points of light following one another in a...,9/30/2013,50.465843,22.891814


## We are now cleaning the data

In [28]:
clean_df = df.copy()

In [29]:
clean_df.isnull().sum()


Date_time                             0
city                                  0
state/province                     5797
country                            9670
UFO_shape                          1932
length_of_encounter_seconds           0
described_duration_of_encounter       0
description                          15
date_documented                       0
latitude                              0
longitude                             0
dtype: int64

In [31]:
clean_df['latitude'] = pd.to_numeric(clean_df['latitude'], errors='coerce')
clean_df['latitude'].dtype

dtype('float64')

In [34]:
clean_df['length_of_encounter_seconds'] = pd.to_numeric(clean_df['length_of_encounter_seconds'], errors='coerce')
clean_df['length_of_encounter_seconds'].dtype

dtype('float64')

In [36]:
clean_df['length_of_encounter_seconds'].isnull().sum()

np.int64(3)

In [37]:
clean_df[
    clean_df["length_of_encounter_seconds"].isnull()
].head(10)

,Date_time,city,state/province,country,UFO_shape,length_of_encounter_seconds,described_duration_of_encounter,description,date_documented,latitude,longitude
27822,7:33:00 PM,bouse,az,us,NaN,NaN,each a few seconds,Driving through Plomosa Pass towards Bouse Loo...,2/16/2000,33.932500,-114.005000
35692,10:52:00 PM,santa cruz,ca,us,NaN,NaN,eight seconds,2 red lights moving together and apart with a ...,4/16/2005,36.974167,-122.029722
58591,1:00:00 PM,ibague (colombia),NaN,NaN,circle,NaN,1/2 segundo,Viajaba a 27.000 pies en un avion comercial ve...,10/30/2006,4.440663,-75.244141


## We are now cleaning the data

In [39]:
text_columns = [
    'city',
    'state/province',
    'country',
    'UFO_shape',
]

for col in text_columns:
    clean_df[col] = (
        clean_df[col].str.strip().str.lower()
    )
clean_df[text_columns].head(10)

,city,state/province,country,UFO_shape
0,san marcos,tx,us,cylinder
1,lackland afb,tx,NaN,light
2,chester (uk/england),NaN,gb,circle
3,edna,tx,us,circle
4,kaneohe,hi,us,light
5,bristol,tn,us,sphere
6,penarth (uk/wales),NaN,gb,circle
7,norwalk,ct,us,disk
8,pell city,al,us,disk
9,live oak,fl,us,disk


In [40]:
clean_df.duplicated().sum()

np.int64(14)

In [41]:
clean_df = clean_df.drop_duplicates()

In [43]:
clean_df["Date_time"] = pd.to_datetime(
    clean_df["Date_time"],
    errors="coerce"
)

clean_df["date_documented"] = pd.to_datetime(
    clean_df["date_documented"],
    errors="coerce"
)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_21128\3559091359.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_df["Date_time"] = pd.to_datetime(
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_21128\3559091359.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_df["date_documented"] = pd.to_datetime(


In [44]:
# Handle missing values based on the cleaning strategy
# Fill missing country values with 'us' (most sightings are US-based)
clean_df['country'].fillna('us', inplace=True)

# Fill missing UFO_shape with 'unknown'
clean_df['UFO_shape'].fillna('unknown', inplace=True)

# Remove rows with missing description (only 15 values, <1%)
clean_df = clean_df.dropna(subset=['description'])

# Remove rows with missing latitude (geographic accuracy)
clean_df = clean_df.dropna(subset=['latitude'])

# Check final missing values
print("Missing values after cleaning:")
print(clean_df.isnull().sum())
print(f"\nDataset shape: {clean_df.shape}")

Missing values after cleaning:
Date_time                             0
city                                  0
state/province                     5792
country                               0
UFO_shape                             0
length_of_encounter_seconds           3
described_duration_of_encounter       0
description                           0
date_documented                       0
latitude                              0
longitude                             0
dtype: int64

Dataset shape: (80302, 11)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_21128\1250669314.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  clean_df['country'].fillna('us', inplace=True)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_21128\1250669314.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_df['country'].fillna('us', inplace=True)
C:\Users\Lenovo\AppData

In [45]:
clean_df.dtypes

Date_time                          datetime64[ns]
city                                       object
state/province                             object
country                                    object
UFO_shape                                  object
length_of_encounter_seconds               float64
described_duration_of_encounter            object
description                                object
date_documented                    datetime64[ns]
latitude                                  float64
longitude                                 float64
dtype: object

In [46]:
clean_df['year'] = clean_df['Date_time'].dt.year
clean_df['month'] = clean_df['Date_time'].dt.month
clean_df['day'] = clean_df['Date_time'].dt.day
clean_df['hour'] = clean_df['Date_time'].dt.hour

In [47]:
clean_df[['Date_time' , 'year' , 'month' , 'day' , 'hour']].head(10)

,Date_time,year,month,day,hour
0,2026-07-16 20:30:00,2026,7,16,20
1,2026-07-16 21:00:00,2026,7,16,21
2,2026-07-16 17:00:00,2026,7,16,17
3,2026-07-16 21:00:00,2026,7,16,21
4,2026-07-16 20:00:00,2026,7,16,20
5,2026-07-16 19:00:00,2026,7,16,19
6,2026-07-16 21:00:00,2026,7,16,21
7,2026-07-16 23:45:00,2026,7,16,23
8,2026-07-16 20:00:00,2026,7,16,20
9,2026-07-16 21:00:00,2026,7,16,21


# We are gonna do some basic exploration

In [49]:
clean_df['country'].value_counts()

country
us    74756
ca     3000
gb     1904
au      538
de      104
Name: count, dtype: int64

In [50]:
clean_df['UFO_shape'].value_counts()

UFO_shape
light        16562
triangle      7862
circle        7606
unknown       7508
fireball      6207
other         5647
sphere        5383
disk          5212
oval          3730
formation     2457
cigar         2057
changing      1962
flash         1328
rectangle     1296
cylinder      1282
diamond       1177
chevron        952
egg            759
teardrop       750
cone           316
cross          233
delta            7
round            2
crescent         2
pyramid          1
flare            1
hexagon          1
dome             1
changed          1
Name: count, dtype: int64

In [51]:
clean_df["city"].value_counts().head(20)

city
seattle                524
phoenix                454
portland               374
las vegas              368
los angeles            352
san diego              338
houston                296
chicago                265
tucson                 241
miami                  239
orlando                220
austin                 218
albuquerque            213
springfield            213
sacramento             201
columbus               200
san jose               189
london (uk/england)    189
san francisco          187
denver                 185
Name: count, dtype: int64

In [52]:
#average duration
clean_df["length_of_encounter_seconds"].mean()

np.float64(9020.29995343653)

In [53]:
#longest encounter
clean_df['length_of_encounter_seconds'].max()


np.float64(97836000.0)

In [54]:
# saving it to a clean dataset

clean_df.to_csv(
    "../data/processed/ufo_clean.csv",
    index=False
)